In [1]:
import os
os.environ["USE_TF"] = "0"

import pandas as pd 
from transformers import AutoTokenizer, TrainingArguments, Trainer, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("2 Project DataSet TEAMWORK/samsum-train.csv")
val_data = pd.read_csv("2 Project DataSet TEAMWORK/samsum-validation.csv")

In [3]:
train_data.isnull().sum()

id          0
dialogue    1
summary     0
dtype: int64

In [4]:
train_data.dropna(inplace=True)

In [5]:
val_data.duplicated().sum()

0

In [6]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


# Data Preprocessing

In [7]:
import re
def clean_data(dialogue):
    dialogue = re.sub(r"\r?\n|\r"," ",dialogue) # remove lines
    dialogue = re.sub(r"\s+"," ",dialogue) # remove extra spaces like tab
    dialogue = re.sub(r"<.*?>"," ",dialogue) # remove html tags
    dialogue = dialogue.strip().lower()
    return dialogue

In [8]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenize the data

In [9]:
%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [10]:
tokenizer = AutoTokenizer.from_pretrained("t5-small")

In [11]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets = tokenizer(data["summary"],padding="max_length",max_length=512,truncation=True)
    inputs["labels"] = targets["input_ids"]
    return inputs

In [12]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [13]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [14]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [15]:
training_args = TrainingArguments(
    output_dir = "./t5_small_finetuned",
    num_train_epochs = 6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps = 500
)

In [16]:
%pip install accelerate

Note: you may need to restart the kernel to use updated packages.


In [17]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [18]:
trainer.train()

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss



KeyboardInterrupt

